# Simulation de bruit réaliste et mitigation

Circuit Bell avec bruit réaliste:
$$|00\rangle \xrightarrow{H \otimes I} \frac{|00\rangle + |10\rangle}{\sqrt{2}} \xrightarrow{\text{CNOT}} \frac{|00\rangle + |11\rangle}{\sqrt{2}}$$

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, Aer, execute
from qiskit.providers.aer.noise import NoiseModel, depolarizing_error, thermal_relaxation_error
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

### Circuit Bell idéal

Prépare $|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}$.

In [ ]:
def bell_circuit():
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)
    qc.measure(0, 0)
    qc.measure(1, 1)
    return qc

qc_ideal = bell_circuit()
qc_ideal.draw('mpl')

### Modèle de bruit réaliste

Paramètres typiques:
- $T_1 = 50\,\mu s$, $T_2 = 30\,\mu s$
- Erreur de dépolarisation $p_{\text{depol}} = 0.001$ par porte
- Durée de porte: $t_g = 100\,ns$

In [ ]:
T1 = 50e-6
T2 = 30e-6
gate_time = 100e-9
p_depol = 0.001

noise_model = NoiseModel()

thermal_error = thermal_relaxation_error(T1, T2, gate_time)
depol_error = depolarizing_error(p_depol, 1)

noise_model.add_all_qubit_quantum_error(thermal_error, ['u1', 'u2', 'u3'])
noise_model.add_all_qubit_quantum_error(
    depol_error @ thermal_error, ['cx']
)

print('Modèle de bruit:')
print(noise_model)

In [ ]:
# Simulation idéale
backend = Aer.get_backend('qasm_simulator')
counts_ideal = execute(qc_ideal, backend, shots=4096).result().get_counts()

# Simulation bruitée
counts_noisy = execute(qc_ideal, backend, shots=4096,
                       noise_model=noise_model).result().get_counts()

print('Idéal:', counts_ideal)
print('Bruité:', counts_noisy)

### Zero-Noise Extrapolation (Mitigation)

Principe: amplifier artificiellement le bruit (facteurs $c=1,2,3$) et extrapoler à $c=0$.

$$\langle O \rangle(c) \approx \langle O \rangle_0 + \alpha c + \beta c^2$$

In [ ]:
def noise_scaling(noise_model, factor):
    nm = NoiseModel()
    for err in noise_model._default_quantum_errors.values():
        for gates, error in err.items():
            scaled = error
            for _ in range(factor - 1):
                scaled = scaled @ error
            nm.add_all_qubit_quantum_error(scaled, list(gates))
    return nm

In [ ]:
factors = [1, 2, 3]
p_00 = []

for c in factors:
    nm_scaled = noise_scaling(noise_model, c)
    counts = execute(qc_ideal, backend, shots=4096,
                     noise_model=nm_scaled).result().get_counts()
    p_00.append(counts.get('00', 0) / 4096)
    print(f'c={c}: P(00) = {p_00[-1]:.4f}')

In [ ]:
# Extrapolation polynomiale degré 2 → c=0
coeffs = np.polyfit(factors, p_00, 2)
p_00_mitigated = np.polyval(coeffs, 0)
p_00_ideal = 0.5

print(f'P(00) idéal:      {p_00_ideal:.4f}')
print(f'P(00) bruité:     {p_00[0]:.4f}')
print(f'P(00) mitigé:     {p_00_mitigated:.4f}')
print(f'Erreur bruitée:   {abs(p_00[0] - p_00_ideal):.4f}')
print(f'Erreur mitigée:   {abs(p_00_mitigated - p_00_ideal):.4f}')

In [ ]:
c_cont = np.linspace(0, 3.5, 100)
fit_curve = np.polyval(coeffs, c_cont)

plt.figure(figsize=(8, 5))
plt.plot(c_cont, fit_curve, 'b-', label='Extrapolation quadratique')
plt.plot(factors, p_00, 'ro', markersize=8, label='Mesures bruitées')
plt.plot(0, p_00_mitigated, 'g*', markersize=15, label='Mitigé (c→0)')
plt.axhline(p_00_ideal, color='k', linestyle='--', label='Idéal')
plt.xlabel('Facteur d\'amplification du bruit c')
plt.ylabel('P(00)')
plt.title('Zero-Noise Extrapolation')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Comparaison finale
labels = ['Idéal', 'Bruité', 'Mitigé ZNE']
values = [p_00_ideal, p_00[0], p_00_mitigated]

plt.bar(labels, values, color=['green', 'red', 'blue'], width=0.5)
plt.ylabel('P(00)')
plt.title('Comparaison: idéal vs bruité vs mitigé')
plt.axhline(0.5, color='k', linestyle='--', alpha=0.5)
plt.grid(True, axis='y', alpha=0.3)
plt.show()

## Questions

**Q1.** Faire varier $T_1$ de 10 à 100 $\mu$s et tracer l'erreur $|P(00)_{\text{bruité}} - P(00)_{\text{idéal}}|$ en fonction de $T_1$. Commenter l'effet du temps de relaxation sur la fidélité.

**Q2.** Implémenter une mitigation par ** Richardson extrapolation ** avec facteurs $c=1,2,3,4$ et comparer l'erreur résiduelle avec l'extrapolation quadratique à 3 points. Laquelle est la plus efficace?

In [ ]:
# Q1: Effet de T1
T1_vals = np.linspace(10e-6, 100e-6, 10)
errors = []

for T1 in T1_vals:
    nm = NoiseModel()
    therm = thermal_relaxation_error(T1, T2, gate_time)
    nm.add_all_qubit_quantum_error(therm, ['u1', 'u2', 'u3'])
    nm.add_all_qubit_quantum_error(depol_error @ therm, ['cx'])
    counts = execute(qc_ideal, backend, shots=4096,
                     noise_model=nm).result().get_counts()
    err = abs(counts.get('00', 0) / 4096 - 0.5)
    errors.append(err)

plt.plot(T1_vals * 1e6, errors, 'o-', linewidth=2)
plt.xlabel('T₁ (μs)')
plt.ylabel('Erreur absolue |P(00) - 0.5|')
plt.title('Effet de T₁ sur la fidélité du circuit Bell')
plt.grid(True)
plt.show()